# VacationPy
---

## Starter Code to Import Libraries and Load the Weather and Coordinates Data

In [27]:
# Dependencies and Setup
import hvplot.pandas
import pandas as pd
import requests

# Import API key
from api_keys import geoapify_key

In [28]:
# Load the CSV file created in Part 1 into a Pandas DataFrame
city_data_df = pd.read_csv("output_data/cities.csv")

# Display sample data
city_data_df.head()

,City_ID,City,Latitude,Longitude,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date
0,0,kingston,17.9970,-76.7936,303.68,62,40,3.09,JM,1738089610
1,1,waitangi,-43.9535,-176.5597,285.83,90,3,2.23,NZ,1738089126
2,2,mount pearl,47.5166,-52.7813,277.22,87,75,10.80,CA,1738089612
3,3,adamstown,-25.0660,-130.1015,298.17,67,0,2.14,PN,1738089080
4,4,port-aux-francais,-49.3500,70.2167,278.44,86,93,12.03,TF,1738089071


---

### Step 1: Create a map that displays a point for every city in the `city_data_df` DataFrame. The size of the point should be the humidity in each city.

In [29]:
# Initialize an empty list to store hotel data
hotel_data = []

# Define the Geoapify API key (must be a string)
geoapify_key = "7ec4a7874e1a4eeda139f5873845dac6"

for index, row in city_data_df.iterrows():
    latitude = row["Latitude"]
    longitude = row["Longitude"]
    
    # Geoapify API request URL with corrected key   #REFEREENCE: CHATGPT
    url = f"https://api.geoapify.com/v2/places?categories=accommodation.hotel&filter=circle:{longitude},{latitude},10000&limit=1&apiKey={geoapify_key}"

# Configure the map plot with valid tiles    #REFEREENCE: CHATGPT
city_map = city_data_df.hvplot.points(
    "Longitude",
    "Latitude",
    geo=True,
    tiles="EsriWorldStreetMap",  
    size="Humidity",
    color="City",
    frame_width=800,
    frame_height=500,
    hover_cols=["City", "Country", "Humidity"]
)

# Display the map
city_map


:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]   (City,Humidity,Country)

### Step 2: Narrow down the `city_data_df` DataFrame to find your ideal weather condition

In [30]:
# Example criteria: Max temperature between 70 and 80 degrees Fahrenheit, and humidity below 60%
ideal_weather_df = city_data_df[
    (city_data_df["Max Temp"] >= 294.15) &  # 294.15K is approximately 70°F
    (city_data_df["Max Temp"] <= 299.15) &  # 299.15K is approximately 80°F
    (city_data_df["Humidity"] < 60)
]

# Drop any rows with null values
ideal_weather_df = ideal_weather_df.dropna()

# Display sample data
ideal_weather_df.head()


,City_ID,City,Latitude,Longitude,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date
105,105,yokadouma,3.5167,15.0500,297.17,27,43,1.13,CM,1738089729
114,114,omdurman,15.6445,32.4777,295.61,15,0,6.17,SD,1738089739
117,117,zouerate,22.7187,-12.4521,296.41,13,0,5.49,MR,1738089743
123,123,jwaneng,-24.6004,24.7303,298.48,30,39,3.46,BW,1738089749
133,133,puerto deseado,-47.7503,-65.8938,297.97,24,14,10.79,AR,1738089761


### Step 3: Create a new DataFrame called `hotel_df`.

In [31]:
# Use the Pandas copy function to create DataFrame called hotel_df
hotel_df = ideal_weather_df[['City', 'Country', 'Latitude', 'Longitude', 'Humidity']].copy()

# Add an empty column, "Hotel Name," to the DataFrame
hotel_df['Hotel Name'] = ""

# Display sample data
hotel_df.head()


,City,Country,Latitude,Longitude,Humidity,Hotel Name
105,yokadouma,CM,3.5167,15.0500,27,
114,omdurman,SD,15.6445,32.4777,15,
117,zouerate,MR,22.7187,-12.4521,13,
123,jwaneng,BW,-24.6004,24.7303,30,
133,puerto deseado,AR,-47.7503,-65.8938,24,


### Step 4: For each city, use the Geoapify API to find the first hotel located within 10,000 metres of your coordinates.

In [34]:
import requests
# Set parameters to search for a hotel
radius = 10000  # 10,000 meters
params = {
    "categories": "accommodation.hotel",
    "limit": 1,
    "apiKey": geoapify_key
}

# Print a message to follow up the hotel search
print("Starting hotel search")

# Iterate through the hotel_df DataFrame
for index, row in hotel_df.iterrows():
    # get latitude, longitude from the DataFrame
    latitude = row["Latitude"]
    longitude = row["Longitude"]

    # Add the current city's latitude and longitude to the params dictionary
    params["filter"] = f"circle:{longitude},{latitude},{radius}"

    # Set base URL
    base_url = "https://api.geoapify.com/v2/places"

    # Make and API request using the params dictionary
    response = requests.get(base_url, params=params)

    # Convert the API response to JSON format
    name_address = response.json()

    # Grab the first hotel from the results and store the name in the hotel_df DataFrame
    try:
        hotel_df.loc[index, "Hotel Name"] = name_address["features"][0]["properties"]["name"]
    except (KeyError, IndexError):
        # If no hotel is found, set the hotel name as "No hotel found".
        hotel_df.loc[index, "Hotel Name"] = "No hotel found"

    # Log the search results
    print(f"{hotel_df.loc[index, 'City']} - nearest hotel: {hotel_df.loc[index, 'Hotel Name']}")

# Display sample data
hotel_df.head()


Starting hotel search
yokadouma - nearest hotel: Hôtel Zokadouma
omdurman - nearest hotel: توتيل للشقق الفندقيه
zouerate - nearest hotel: فندق تيرس زمور
jwaneng - nearest hotel: Mokala Lodge & Teemane Casino
puerto deseado - nearest hotel: Las Bandurrias
partur - nearest hotel: No hotel found
bako - nearest hotel: Jinka Resort
sur - nearest hotel: Resort Sur Beach Hotel
ixtapa - nearest hotel: Krystal Puerto Vallarta
taoudenni - nearest hotel: No hotel found
nguigmi - nearest hotel: Guest PAM
gao - nearest hotel: No hotel found
alaghsas - nearest hotel: Hôtel de l’AÏR
goure - nearest hotel: No hotel found
fontem - nearest hotel: No hotel found
ouadda - nearest hotel: No hotel found
opi - nearest hotel: No hotel found
tchintabaraden - nearest hotel: No hotel found
sarh - nearest hotel: فندق سفاري
reggane - nearest hotel: No hotel found
vredendal - nearest hotel: No hotel found
mahalapye - nearest hotel: Cresta Mahalapye
arauco - nearest hotel: No hotel found
wolmaransstad - nearest hote

,City,Country,Latitude,Longitude,Humidity,Hotel Name
105,yokadouma,CM,3.5167,15.0500,27,Hôtel Zokadouma
114,omdurman,SD,15.6445,32.4777,15,توتيل للشقق الفندقيه
117,zouerate,MR,22.7187,-12.4521,13,فندق تيرس زمور
123,jwaneng,BW,-24.6004,24.7303,30,Mokala Lodge & Teemane Casino
133,puerto deseado,AR,-47.7503,-65.8938,24,Las Bandurrias


### Step 5: Add the hotel name and the country as additional information in the hover message for each city in the map.

In [37]:
import hvplot.pandas  # Ensure hvplot is imported

# Configure the map plot
city_map = hotel_df.hvplot.points(
    "Longitude",
    "Latitude",
    geo=True,
    tiles='EsriWorldStreetMap',  
    size=20,  
    color='City',  
    frame_width=800,
    frame_height=500,
    hover_cols=['City', 'Country', 'Hotel Name']  #
)

# Display the map
city_map


:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]   (City,Country,Hotel Name)